# Explorative Analyse: Arbeitsmarkt Regensburg (Jobcenter) — Teil 1: Exploration

**Datenquelle:** Statistik der Bundesagentur für Arbeit, Bericht *"Eckwerte für Jobcenter"*, Jobcenter Regensburg
(Statistik-Nr. t73906-0), monatliche Excel-Berichte aus `Data/RegensburgJCData/`.

**Zeitraum:** Januar 2025 – Juni 2026 (18 Berichtsmonate, ein `.xlsx` je Monat).

Jeder Bericht enthält u. a. ein Tabellenblatt **"1.1 Eckwerte"** mit dem aktuellen Monatswert je Merkmal,
getrennt nach *"Insgesamt (SGB II und SGB III)"* und *"Rechtskreis SGB II"* (= Jobcenter-Zuständigkeit).

Dieses Notebook kümmert sich um **Einlesen, Bereinigen und Aufbereiten** der Daten und exportiert am Ende
zwei aufbereitete CSV-Dateien. Die eigentliche **Visualisierung und Auswertung** erfolgt in
[`02_Analyse_SGBII.ipynb`](02_Analyse_SGBII.ipynb).

In [2]:
import re
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd

DATA_DIR = Path("..") / "Data" / "RegensburgJCData"
PROCESSED_DIR = Path("..") / "Data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)

## 1. Tabellenblätter der Rohdaten untersuchen

Jede Excel-Datei enthält nicht nur "1.1 Eckwerte", sondern rund 20 Tabellenblätter — u. a. mit
Zahlen zu **SGB III** (Arbeitslosenversicherung, Agentur für Arbeit), die bislang komplett
außen vor gelassen wurden. Bevor wir uns auf "1.1 Eckwerte" festlegen, schauen wir uns alle
Blätter an: was steht drin, ist es für die SGB-II-Arbeitslosenanalyse relevant, und welche
Diagramme ließen sich daraus bauen?

**Wichtiger Befund vorab:** Die Blätter **3.1–3.6** und **4.1–4.2** melden nicht den aktuellen
Berichtsmonat, sondern Daten mit **3 bzw. 6 Monaten Wartezeit** (z. B. "März 2026" oder
"Dezember 2025", obwohl der Bericht selbst "Juni 2026" heißt). Wer daraus eine Zeitreihe über
mehrere Dateien bauen will, muss das beim Zuordnen des Berichtsmonats berücksichtigen.

In [3]:
sheet_notes = {
    "Deckblatt": (
        "Nur Titelseite, keine Zahlen.",
        "Kein Diagramm möglich.",
    ),
    "Impressum": (
        "Metadaten zum Bericht selbst (Titel, Region, Erstellungs- und Veröffentlichungsdatum, "
        "Rückfragen-Kontakt). Keine Arbeitsmarktdaten.",
        "Kein Diagramm möglich.",
    ),
    "Inhaltsverzeichnis": (
        "Übersicht aller Blätter plus Zeichenerklärung (z. B. 'x' = keine Berechnung möglich, "
        "'*' = aus Datenschutzgründen anonymisiert). Nützlich als Nachschlagewerk beim Parsen "
        "anderer Blätter.",
        "Kein Diagramm möglich.",
    ),
    "Grafik": (
        "Enthält nur eingebettete Bild-Diagramme der BA selbst, keine auslesbaren Tabellenwerte.",
        "Für uns nicht nutzbar (kein Zahlenzugriff über openpyxl).",
    ),
    "1.1 Eckwerte": (
        "Bereits unsere Hauptquelle. Enthält über die bisher genutzten Zeilen hinaus auch "
        "Zugang/Abgang (im Monat + 12-Monatssumme), gemeldete Arbeitsstellen und eine quartalsweise "
        "Beschäftigtenstatistik (sozialversicherungspflichtig / geringfügig, nach Arbeits- und "
        "Wohnort) — bislang ungenutzt.",
        "Zugang-vs-Abgang als Balken- oder Flussdiagramm; gemeldete Arbeitsstellen als Nachfrage-"
        "Indikator neben der Arbeitslosenzahl.",
    ),
    "1.2 Eckwerte Zeitreihe": (
        "Enthält bereits eine rollierende ~13-Monats-Zeitreihe direkt im Bericht (gleiche Kennzahlen "
        "wie 1.1). Könnte als Cross-Check für unsere selbst gebaute Zeitreihe dienen.",
        "Kein neues Diagramm, aber Validierungsmöglichkeit für die Übersichtsgrafik.",
    ),
    "2.1 Arbeitslosigkeit Zugang": (
        "Zugang an Arbeitslosen nach Zugangsgrund (z. B. 'aus Erwerbstätigkeit', vermutlich auch "
        "aus Ausbildung/Nichterwerbstätigkeit weiter unten im Blatt) sowie gleitende 12-Monatssumme.",
        "Gestapeltes Balkendiagramm: woher kommen die neuen Arbeitslosen?",
    ),
    "2.2 Arbeitslosigkeit Bestand": (
        "Bestand an Arbeitslosen nach Personengruppen mit Anteilsspalte (%) — inhaltlich weitgehend "
        "deckungsgleich mit dem, was wir schon aus 1.1 extrahieren.",
        "Kein großer Mehrwert gegenüber der Übersichtsgrafik.",
    ),
    "2.3 Arbeitslosigkeit Abgang": (
        "Abgang an Arbeitslosen nach Abgangsgrund (z. B. 'in Erwerbstätigkeit').",
        "Zugang vs. Abgang nach Grund gegenüberstellen, um zu sehen, wie viele tatsächlich in "
        "Arbeit abgehen statt z. B. in Nichterwerbstätigkeit.",
    ),
    "2.4 Langzeitarbeitslosigkeit": (
        "Langzeitarbeitslose nach Personengruppen (Geschlecht, Alter) mit Anteilsspalte — dieselbe "
        "Kennzahl wie unser 'Langzeitarbeitslose', aber zusätzlich nach Geschlecht aufgeschlüsselt.",
        "Geschlechterverteilung der Langzeitarbeitslosen als Zeitreihe (aktuell haben wir das nur "
        "als Momentaufnahme für alle Arbeitslosen, nicht speziell für LZA).",
    ),
    "2.5 Unterbeschäftigung": (
        "Feinere Komponenten der Unterbeschäftigungs-Lücke (z. B. 'Aktivierung und berufliche "
        "Eingliederung', 'Sonderregelung für Ältere').",
        "Gestapeltes Flächendiagramm: woraus setzt sich die Differenz zwischen Arbeitslosigkeit "
        "und Unterbeschäftigung zusammen?",
    ),
    "3.1 Bedarfsgemeinschaften": (
        "Struktur der Bedarfsgemeinschaften nach Haushaltsgröße (1, 2, 3, 4+ Personen). "
        "⚠️ 3 Monate Wartezeit.",
        "Balkendiagramm der Haushaltsgrößenverteilung; Zeitreihe des Single-Haushalt-Anteils.",
    ),
    "3.2 Personen in BG": (
        "Personen in BG nach Geschlecht, plus Teilmenge ELB — ergänzt unsere PERS/BG-Zeitreihe um "
        "eine Geschlechteraufschlüsselung. ⚠️ 3 Monate Wartezeit.",
        "Geschlechterverteilung der Personen in BG als Zeitreihe.",
    ),
    "3.3 Erwerbstätigkeit": (
        "'Aufstocker' (erwerbstätige ELB) — Personen mit Job, die trotzdem ergänzend Bürgergeld "
        "beziehen, aufgeschlüsselt nach abhängig/selbstständig und Einkommenshöhe. Komplett neues "
        "Thema, direkt anschlussfähig an unsere Trichter-Grafik. ⚠️ 3 Monate Wartezeit.",
        "Zeitreihe der Aufstocker-Zahl; zusätzliche Trichter-Stufe 'davon in Arbeit, aber bedürftig'.",
    ),
    "3.4 Langzeitleistungsbezug": (
        "Neue Kennzahl: Langzeitleistungsbeziehende (LZB, mind. 21 von 24 Monaten im Bezug) — "
        "ähnlich, aber nicht identisch mit Langzeitarbeitslosen, da Bürgergeld auch ohne "
        "Arbeitslosigkeit bezogen werden kann (z. B. bei Aufstockern). ⚠️ 3 Monate Wartezeit.",
        "LZB vs. LZA im Liniendiagramm vergleichen, um zu zeigen, dass Bedürftigkeit oft länger "
        "anhält als reine Arbeitslosigkeit.",
    ),
    "3.5 Bewegungen Personen": (
        "Zu-/Abgänge im Regelleistungsbezug (ELB-Ebene) inkl. Vorbezugs-Historie (z. B. Anteil mit "
        "erneutem Bezug innerhalb der letzten 12 Monate — Hinweis auf 'Drehtür-Effekt'). "
        "⚠️ 3 Monate Wartezeit.",
        "Anteil der Wiederholungsfälle als Zeitreihe.",
    ),
    "3.6 Zahlungsansprüche": (
        "Einzige Quelle für **Euro-Beträge**: Summe der Zahlungsansprüche, Ø Anspruch je BG, "
        "aufgeschlüsselt nach Regelbedarf ELB/NEF. Bisher komplett unberührtes Themenfeld "
        "(Kostenvolumen). ⚠️ 3 Monate Wartezeit.",
        "Zeitreihe der monatlichen Gesamtsumme in Euro oder des Ø Anspruchs je BG.",
    ),
    "4.1 Förderung": (
        "Eintritte in arbeitsmarktpolitische Maßnahmen nach Instrumenten-Kategorie inkl. gleitender "
        "12-Monatssumme. Komplett neues Themenfeld (aktive Arbeitsmarktpolitik). "
        "⚠️ 3 Monate Wartezeit.",
        "Balkendiagramm der Maßnahmen-Kategorien nach Teilnehmerzahl.",
    ),
    "4.2 Förderung Strukturen": (
        "Bestand an Maßnahmenteilnehmenden nach Geschlecht, Alter, LZA-Status, Schwerbehinderung; "
        "enthält außerdem die fertige Kennzahl Aktivierungsquote (AQ1/AQ2a). "
        "⚠️ 3 Monate Wartezeit.",
        "Gestapeltes Balkendiagramm nach Altersgruppe; Aktivierungsquote als Zeitreihe.",
    ),
    "Linkliste": (
        "Nur externe Web-Links zu weiterführenden BA-Statistikseiten, keine Zahlen.",
        "Kein Diagramm möglich.",
    ),
    "Statistik-Infoseite": (
        "Nur eine Themen-Übersichtsseite mit Verweisen auf andere BA-Statistikbereiche, keine Zahlen.",
        "Kein Diagramm möglich.",
    ),
}

files = sorted(DATA_DIR.glob("*.xlsx"))
beispiel_datei = files[-1]
wb_sample = openpyxl.load_workbook(beispiel_datei, data_only=True)
print(f"Untersuchtes Beispiel: {beispiel_datei.name}")
print(f"{len(wb_sample.sheetnames)} Tabellenblätter gefunden.\n")

for name in wb_sample.sheetnames:
    ws = wb_sample[name]
    daten, diagramme = sheet_notes.get(name, ("(keine Einschätzung hinterlegt)", "-"))
    print(f"--- {name} (dims={ws.dimensions}) ---")
    print(f"Daten: {daten}")
    print(f"Diagramme: {diagramme}")
    print()

C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:33: UserWarning: DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.
  warn("DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.")
C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


Untersuchtes Beispiel: jc-eckwerte-t73906-0-202606-xlsx.xlsx
21 Tabellenblätter gefunden.

--- Deckblatt (dims=A1:A1) ---
Daten: Nur Titelseite, keine Zahlen.
Diagramme: Kein Diagramm möglich.

--- Impressum (dims=A1:F39) ---
Daten: Metadaten zum Bericht selbst (Titel, Region, Erstellungs- und Veröffentlichungsdatum, Rückfragen-Kontakt). Keine Arbeitsmarktdaten.
Diagramme: Kein Diagramm möglich.

--- Inhaltsverzeichnis (dims=A1:G50) ---
Daten: Übersicht aller Blätter plus Zeichenerklärung (z. B. 'x' = keine Berechnung möglich, '*' = aus Datenschutzgründen anonymisiert). Nützlich als Nachschlagewerk beim Parsen anderer Blätter.
Diagramme: Kein Diagramm möglich.

--- Grafik (dims=A1:L58) ---
Daten: Enthält nur eingebettete Bild-Diagramme der BA selbst, keine auslesbaren Tabellenwerte.
Diagramme: Für uns nicht nutzbar (kein Zahlenzugriff über openpyxl).

--- 1.1 Eckwerte (dims=A1:K70) ---
Daten: Bereits unsere Hauptquelle. Enthält über die bisher genutzten Zeilen hinaus auch Zugang/Abgang

## 2. Daten einlesen

Alle Monatsberichte parsen und in ein *tidy* DataFrame (`period`, `kategorie`, `merkmal`, `insgesamt`, `sgb2`) überführen.

In [4]:
def normalize_label(label: str) -> str:
    """Entfernt Fußnotenmarker wie ' 1)' oder ' 2) 4)' aus Zeilenbeschriftungen."""
    return re.sub(r"\s*\d\)", "", label).strip()


def parse_eckwerte(path: Path) -> list[dict]:
    """Liest das Blatt '1.1 Eckwerte' eines Monatsberichts aus.

    Spalte B enthält den Wert 'Insgesamt (SGB II und SGB III)', Spalte G den
    Wert für den 'Rechtskreis SGB II' (= Jobcenter) im aktuellen Berichtsmonat.
    Zeilen ohne Zahlwert sind Kategorie-Überschriften (z.B. 'Arbeitslose').
    """
    match = re.search(r"(\d{6})-xlsx\.xlsx$", path.name)
    period = pd.Period(match.group(1), freq="M")

    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["1.1 Eckwerte"]

    records = []
    category = None
    for row in range(9, 48):
        label = ws.cell(row=row, column=1).value
        insgesamt = ws.cell(row=row, column=2).value
        sgb2 = ws.cell(row=row, column=7).value

        if not isinstance(label, str) or not label.strip():
            continue
        label = normalize_label(label)

        is_value_row = isinstance(insgesamt, (int, float)) or isinstance(sgb2, (int, float))
        if not is_value_row:
            category = label
            continue

        records.append({
            "period": period,
            "kategorie": category,
            "merkmal": label,
            "insgesamt": insgesamt,
            "sgb2": sgb2,
        })
    return records


files = sorted(DATA_DIR.glob("*.xlsx"))
print(f"{len(files)} Berichte gefunden: {files[0].name} ... {files[-1].name}")

all_records = [rec for f in files for rec in parse_eckwerte(f)]
df = pd.DataFrame(all_records).sort_values(["period", "kategorie", "merkmal"]).reset_index(drop=True)
df["jahr"] = df["period"].dt.year
df["monat"] = df["period"].dt.month
df["label"] = df["kategorie"] + " | " + df["merkmal"]
df.head(5)

18 Berichte gefunden: jc-eckwerte-t73906-0-202501-xlsx.xlsx ... jc-eckwerte-t73906-0-202606-xlsx.xlsx


,period,kategorie,merkmal,insgesamt,sgb2,jahr,monat,label
0,2025-01,Arbeitslose,15 bis unter 25 Jahre,373.0,148.0,2025,1,Arbeitslose | 15 bis unter 25 Jahre
1,2025-01,Arbeitslose,25 bis unter 50 Jahre,1900.0,848.0,2025,1,Arbeitslose | 25 bis unter 50 Jahre
2,2025-01,Arbeitslose,50 Jahre und älter,1470.0,474.0,2025,1,Arbeitslose | 50 Jahre und älter
3,2025-01,Arbeitslose,55 Jahre und älter,1114.0,307.0,2025,1,Arbeitslose | 55 Jahre und älter
4,2025-01,Arbeitslose,Abgang (12-Monatssumme),9197.0,3318.0,2025,1,Arbeitslose | Abgang (12-Monatssumme)


## 3. Datenüberblick und Bereinigung

In [5]:
print("Shape:", df.shape)
print("Berichtsmonate:", df['period'].min(), "-", df['period'].max(), f"({df['period'].nunique()} Monate)")
print("Duplikate:", df.duplicated().sum())
print("Fehlende Werte (sgb2):", df['sgb2'].isna().sum(), "von", len(df))
print("\nMerkmale je Kategorie:")
for kat, sub in df.groupby("kategorie", sort=False):
    print(f"- {kat}: {sorted(sub['merkmal'].unique())}")

Shape: (558, 8)
Berichtsmonate: 2025-01 - 2026-06 (18 Monate)
Duplikate: 0
Fehlende Werte (sgb2): 108 von 558

Merkmale je Kategorie:
- Arbeitslose: ['15 bis unter 25 Jahre', '25 bis unter 50 Jahre', '50 Jahre und älter', '55 Jahre und älter', 'Abgang (12-Monatssumme)', 'Abgang (im Monat)', 'Arbeitslosenquote', 'Ausländer', 'Bestand', 'Frauen', 'Langzeitarbeitslose', 'Männer', 'Zugang (12-Monatssumme)', 'Zugang (im Monat)', 'schwerbehinderte Menschen']
- Arbeitsuchende: ['Bestand']
- Grundsicherung für Arbeitsuchende: ['Bedarfsgemeinschaften (BG)', 'Personen in Bedarfsgemeinschaften (PERS)', 'dar. Regelleistungsberechtigte (RLB)', 'dav. erwerbsfähige Leistungsberechtigte (ELB)', 'nicht erwerbsfähige Leistungsberechtigte (NEF)']
- Unterbeschäftigung: ['Arbeitslosigkeit im weiteren Sinne', 'Unterbeschäftigung (ohne Kurzarbeit)', 'Unterbeschäftigung im engeren Sinne', 'Unterbeschäftigungsquote']
- gemeldete Arbeitsstellen: ['Bestand', 'Zugang (12-Monatssumme)', 'Zugang (im Monat)', 'sozia

In [6]:
print("Kennzahlen der SGB-II-Werte (deskriptive Statistik):")
df["sgb2"].describe()

Kennzahlen der SGB-II-Werte (deskriptive Statistik):


count     450.000000
mean     1596.854368
std      1511.517100
min         1.200000
25%       332.000000
50%       862.000000
75%      2672.335983
max      5488.033274
Name: sgb2, dtype: float64

## 4. Kennzahlen-Zeitreihe aufbereiten

Die für die Analyse relevanten Merkmale werden aus dem langen Format per `pivot_table()` in eine breite
Zeitreihen-Tabelle (eine Zeile je Monat, eine Spalte je Kennzahl) überführt.

In [7]:
kennzahlen_labels = {
    "Arbeitslose | Bestand": "Arbeitslose (Bestand)",
    "Arbeitslose | Arbeitslosenquote": "Arbeitslosenquote (%)",
    "Arbeitslose | Langzeitarbeitslose": "Langzeitarbeitslose",
    "Unterbeschäftigung | Unterbeschäftigungsquote": "Unterbeschäftigungsquote (%)",
    "Grundsicherung für Arbeitsuchende | Bedarfsgemeinschaften (BG)": "Bedarfsgemeinschaften",
    "Grundsicherung für Arbeitsuchende | Personen in Bedarfsgemeinschaften (PERS)": "Personen in BG",
    "Grundsicherung für Arbeitsuchende | dar. Regelleistungsberechtigte (RLB)": "Regelleistungsberechtigte (RLB)",
    "Grundsicherung für Arbeitsuchende | dav. erwerbsfähige Leistungsberechtigte (ELB)": "Erwerbsfähige Leistungsberechtigte (ELB)",
    "Grundsicherung für Arbeitsuchende | nicht erwerbsfähige Leistungsberechtigte (NEF)": "Nicht erwerbsfähige Leistungsberechtigte (NEF)",
}

subset = df[df["label"].isin(kennzahlen_labels)].copy()
subset["kennzahl"] = subset["label"].map(kennzahlen_labels)

kennzahlen = subset.pivot_table(index="period", columns="kennzahl", values="sgb2")
kennzahlen = kennzahlen[list(kennzahlen_labels.values())]
kennzahlen.index = kennzahlen.index.to_timestamp()

print("Mittelwert / Std je Kennzahl (NumPy):")
for spalte in kennzahlen.columns:
    werte = kennzahlen[spalte].to_numpy()
    print(f"- {spalte}: mean={np.mean(werte):.1f}, std={np.std(werte):.1f}")

kennzahlen.head()

Mittelwert / Std je Kennzahl (NumPy):
- Arbeitslose (Bestand): mean=1504.0, std=56.8
- Arbeitslosenquote (%): mean=1.3, std=0.1
- Langzeitarbeitslose: mean=602.2, std=68.8
- Unterbeschäftigungsquote (%): mean=1.8, std=0.1
- Bedarfsgemeinschaften: mean=2691.7, std=54.1
- Personen in BG: mean=5247.4, std=124.0
- Regelleistungsberechtigte (RLB): mean=4950.6, std=124.4
- Erwerbsfähige Leistungsberechtigte (ELB): mean=3584.4, std=82.5
- Nicht erwerbsfähige Leistungsberechtigte (NEF): mean=1366.2, std=44.9


kennzahl,Arbeitslose (Bestand),Arbeitslosenquote (%),Langzeitarbeitslose,Unterbeschäftigungsquote (%),Bedarfsgemeinschaften,Personen in BG,Regelleistungsberechtigte (RLB),Erwerbsfähige Leistungsberechtigte (ELB),Nicht erwerbsfähige Leistungsberechtigte (NEF)
period,,,,,,,,,
2025-01-01,1470.0,1.3,502.0,1.9,2797.303381,5488.033274,5202.270477,3730.590852,1471.679625
2025-02-01,1382.0,1.2,502.0,1.8,2755.768665,5434.992911,5147.992911,3710.211568,1437.781343
2025-03-01,1414.0,1.2,513.0,1.8,2781.436728,5474.933615,5178.933615,3746.433943,1432.499672
2025-04-01,1457.0,1.3,526.0,1.8,2766.959069,5421.148834,5122.424563,3701.239670,1421.184893
2025-05-01,1438.0,1.2,558.0,1.8,2707.556182,5295.629265,4989.629265,3612.418872,1377.210393


## 5. Export für die Analyse

Beide DataFrames werden als CSV abgelegt, damit `02_Analyse_SGBII.ipynb` unabhängig davon starten kann.

In [8]:
df.to_csv(PROCESSED_DIR / "eckwerte_long.csv", index=False)
kennzahlen.to_csv(PROCESSED_DIR / "kennzahlen_sgb2.csv", index_label="period")
print("Exportiert nach:", PROCESSED_DIR.resolve())

Exportiert nach: C:\Users\funke\OneDrive\Skills_Update\arbeitsmarkt-regensburg\Data\processed


Weiter geht es in [`02_Analyse_SGBII.ipynb`](02_Analyse_SGBII.ipynb) mit den Visualisierungen und der inhaltlichen Auswertung.